# Predicting Mortality of Heart Failure Patients
### Intermediate Level | Machine Learning on Real Clinical Data

This project builds a complete, real machine learning pipeline on real clinical data: predicting whether a heart failure patient survives or dies during their follow-up period, using 12 clinical features recorded at the time of diagnosis.

1. **Section 1 — Imports and Setup**
2. **Section 2 — Data Extraction and Verification**
3. **Section 3 — Exploratory Data Analysis (EDA)**
4. **Section 4 — Preprocessing**
5. **Section 5 — Building and Training the Models**
6. **Section 6 — Evaluation**
7. **Section 7 — Feature Importance and Interpretability**
8. **Section 8 — Testing on Individual Patients**

**About the dataset:** 299 patients treated for heart failure at the Faisalabad Institute of Cardiology and the Allied Hospital in Faisalabad, Pakistan, during 2015 (Ahmad et al., 2017). Each row is one patient's clinical profile at the start of the follow-up period; the target, `DEATH_EVENT`, records whether that patient died before the follow-up period ended.


## Why This Matters

Cardiovascular disease is the leading cause of death worldwide, and heart failure is one of its most common and severe forms. Being able to estimate a patient's mortality risk from routine clinical measurements — without invasive or expensive tests — can help clinicians prioritize care and monitoring.

This is also a good project for learning classical machine learning end to end: real (if modestly sized) clinical data, class imbalance, the need for interpretability, and a comparison between a simple, transparent model and a more powerful one.


## Section 1 — Imports and Setup


In [ ]:
# This checks what's already installed and only installs what's missing -
# much faster than a blind "pip install" every time.
import importlib
import subprocess
import sys

required_packages = [
    "requests",     # for downloading the dataset with proper error handling
    "pandas",       # tabular data handling
    "numpy",        # array math
    "matplotlib",   # plotting
    "seaborn",      # statistical visualizations (correlation heatmaps, etc.)
    "scikit-learn", # models, preprocessing, evaluation metrics
]

missing = []
for package_name in required_packages:
    try:
        importlib.metadata.version(package_name)  # checks metadata only - does not import the package
    except importlib.metadata.PackageNotFoundError:
        missing.append(package_name)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Done.")
else:
    print("All required packages are already installed - nothing to do.")


In [ ]:
import numpy as np  # array math
import pandas as pd  # loading and manipulating the tabular dataset
import matplotlib.pyplot as plt  # plotting distributions and evaluation charts
import seaborn as sns  # nicer statistical plots (correlation heatmap, boxplots)

from sklearn.model_selection import train_test_split  # splits data into training and test sets
from sklearn.preprocessing import StandardScaler  # scales features to comparable ranges
from sklearn.linear_model import LogisticRegression  # our simple, interpretable baseline model
from sklearn.ensemble import RandomForestClassifier  # our more powerful, ensemble model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,  # core classification metrics
    confusion_matrix, classification_report, roc_curve, roc_auc_score,  # richer evaluation tools
)

sns.set_style("whitegrid")  # a clean, readable default style for every plot in this notebook
print("Libraries loaded.")


## Section 2 — Data Extraction and Verification

We download the real dataset directly as a CSV. Two independent mirrors are tried in sequence in case one is temporarily unavailable, and the result is verified against the dataset's known, published shape before we proceed — so a data problem is caught here, not several cells later as a confusing model result.


In [ ]:
DATA_URLS = [
    "https://raw.githubusercontent.com/lorenzodenisi/Heart-Failure-Clinical-Records/master/heart_failure_clinical_records_dataset.csv",
    "https://raw.githubusercontent.com/RanjanRavi2398/Heart-Failure-Using-Clinical-Record-Set/master/heart_failure_clinical_records_dataset.csv",
]


def load_dataset():
    """Tries each mirror in turn and returns the first one that loads
    successfully as a proper CSV."""
    import requests
    import io

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

    for url in DATA_URLS:
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text))
            print(f"Loaded successfully from: {url}")
            return df
        except Exception as e:
            print(f"Failed to load from {url}: {e}")

    return None


df = load_dataset()

if df is None:
    print()
    print("MANUAL FALLBACK: download the dataset yourself from one of these sources:")
    print("  - UCI: https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records")
    print("  - Kaggle: https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data")
    print("Then place the CSV in the same folder as this notebook and load it with:")
    print('  df = pd.read_csv("heart_failure_clinical_records_dataset.csv")')
    raise RuntimeError("Could not load the dataset automatically - see manual fallback above.")


In [ ]:
# Verification step: confirm the data matches the dataset's known, published
# shape and structure before doing anything else with it.
print("Shape:", df.shape)
assert df.shape == (299, 13), f"Expected 299 rows and 13 columns, got {df.shape}"

print("Missing values:", df.isnull().sum().sum())
assert df.isnull().sum().sum() == 0, "Expected no missing values in this dataset"

print("Columns:", list(df.columns))
expected_columns = {
    "age", "anaemia", "creatinine_phosphokinase", "diabetes", "ejection_fraction",
    "high_blood_pressure", "platelets", "serum_creatinine", "serum_sodium",
    "sex", "smoking", "time", "DEATH_EVENT",
}
assert set(df.columns) == expected_columns, "Column names don't match the expected dataset"

print()
print("Verification passed - this is the real, complete dataset.")
df.head()


## Section 3 — Exploratory Data Analysis

Before building any model, we look at the data itself: how many patients survived versus died, how the key clinical features differ between those two groups, and how features relate to each other.


In [ ]:
# Class balance: how many patients died (1) versus survived (0)?
death_counts = df["DEATH_EVENT"].value_counts()
print(death_counts)
print(f"\nMortality rate: {df['DEATH_EVENT'].mean():.1%}")

plt.figure(figsize=(5, 4))
sns.countplot(x="DEATH_EVENT", data=df, hue="DEATH_EVENT", palette=["#4C72B0", "#C44E52"], legend=False)
plt.xticks([0, 1], ["Survived", "Died"])
plt.title("Class Balance: Survived vs Died")
plt.ylabel("Number of patients")
plt.xlabel("")
plt.show()


In [ ]:
# Compare a few key clinical features between patients who survived and
# patients who died - this is often where the most useful signal is visible
# before any model is even built.
key_features = ["age", "ejection_fraction", "serum_creatinine", "time"]

fig, axes = plt.subplots(1, len(key_features), figsize=(20, 4))
for ax, feature in zip(axes, key_features):
    sns.boxplot(x="DEATH_EVENT", y=feature, data=df, hue="DEATH_EVENT", palette=["#4C72B0", "#C44E52"], legend=False, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Survived", "Died"])
    ax.set_title(feature)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap: which features move together, and which correlate
# most strongly with the outcome itself?
plt.figure(figsize=(11, 9))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

print("Features most correlated with DEATH_EVENT:")
print(correlation_matrix["DEATH_EVENT"].abs().sort_values(ascending=False)[1:6])


## Section 4 — Preprocessing

We split the data into training and test sets before doing anything else, so the test set stays a genuinely unseen evaluation of the final model. Because roughly a third of patients died, we use **stratified** splitting to keep that same ratio in both sets. We also scale the features - important for logistic regression, since features here span very different ranges (age in years, platelets in the hundreds of thousands).


In [ ]:
X = df.drop(columns=["DEATH_EVENT"])  # all clinical features
y = df["DEATH_EVENT"]                   # the target we're predicting

# stratify=y keeps the same survived/died ratio in both the training and test sets,
# which matters here since the classes are not evenly balanced.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, "  Test set:", X_test.shape)
print("Training set mortality rate:", y_train.mean().round(3))
print("Test set mortality rate:", y_test.mean().round(3))


In [ ]:
# Scale features to have zero mean and unit variance. We fit the scaler on
# the TRAINING data only, then apply that same transformation to the test
# set - fitting on the test set too would leak information about it into
# the model indirectly.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. Example - before vs after for the first patient's age:")
print("Before:", X_train.iloc[0]["age"], " After:", round(X_train_scaled[0][0], 3))


## Section 5 — Building and Training the Models

We train two models with very different strengths, so we can compare them directly:
- **Logistic Regression** — simple, fast, and directly interpretable (its coefficients tell us how each feature pushes the prediction).
- **Random Forest** — an ensemble of decision trees, usually more accurate on this kind of tabular data, and gives us feature importances for interpretability too.


In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)  # logistic regression needs scaled features to train properly
print("Logistic Regression trained.")

random_forest = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
random_forest.fit(X_train, y_train)  # tree-based models don't need feature scaling
print("Random Forest trained.")


## Section 6 — Evaluation

We evaluate both models on the held-out test set with several metrics, since accuracy alone can be misleading on an imbalanced dataset like this one (a model that always predicts "survived" would already be ~68% accurate without learning anything useful).


In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"F1 score:  {f1_score(y_true, y_pred):.3f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_proba):.3f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["Survived", "Died"]))
    print()


log_reg_pred = log_reg.predict(X_test_scaled)
log_reg_proba = log_reg.predict_proba(X_test_scaled)[:, 1]  # probability of "died"

rf_pred = random_forest.predict(X_test)
rf_proba = random_forest.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression", y_test, log_reg_pred, log_reg_proba)
evaluate_model("Random Forest", y_test, rf_pred, rf_proba)


In [ ]:
# Confusion matrices - a direct look at which mistakes each model makes:
# false positives (predicted death, patient survived) vs false negatives
# (predicted survival, patient died) - in a clinical setting these are not
# equally costly, which is worth discussing explicitly.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, name, pred in zip(axes, ["Logistic Regression", "Random Forest"], [log_reg_pred, rf_pred]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Survived", "Died"], yticklabels=["Survived", "Died"], ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# ROC curves - how well each model separates the two classes across every
# possible decision threshold, not just the default 0.5 cutoff.
plt.figure(figsize=(6, 6))

for name, proba in [("Logistic Regression", log_reg_proba), ("Random Forest", rf_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## Section 7 — Feature Importance and Interpretability

In a clinical setting, *why* a model makes a prediction often matters as much as the prediction itself. Both models here can tell us which features drive their decisions, though in different ways.


In [ ]:
# Random Forest feature importance: how much each feature reduced impurity
# across all the trees in the forest, on average.
importances = pd.Series(random_forest.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette="viridis", legend=False)
plt.title("Random Forest - Feature Importance")
plt.xlabel("Importance")
plt.show()

print(importances)


In [ ]:
# Logistic Regression coefficients: the sign tells us the DIRECTION of the
# effect (positive = pushes toward "died"), and the magnitude (on scaled
# features) is roughly comparable across features.
coefficients = pd.Series(log_reg.coef_[0], index=X.columns).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#C44E52" if c > 0 else "#4C72B0" for c in coefficients.values]
plt.barh(coefficients.index, coefficients.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression Coefficients (red = increases death risk)")
plt.xlabel("Coefficient (on scaled features)")
plt.show()


## Section 8 — Testing on Individual Patients

Aggregate metrics are useful, but it's worth looking at a few individual predictions directly - this is closer to how a model like this would actually be used in practice, one patient at a time.


In [ ]:
def predict_patient(patient_row, model, scaled=False):
    """Runs a model on a single patient (given as a row from X_test) and
    prints a readable summary of the prediction."""
    # Keep this as a one-row DataFrame (not a plain numpy array) so the
    # model sees the same named columns it was trained on.
    features = patient_row.to_frame().T
    if scaled:
        features = scaler.transform(features)

    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]  # probability of "died"

    outcome = "DIED" if prediction == 1 else "SURVIVED"
    print(f"Predicted: {outcome}  (probability of death: {probability:.2f})")


# Try a few real patients from the test set and compare against their actual outcome.
sample_indices = X_test.index[:5]

for idx in sample_indices:
    actual = "DIED" if y_test.loc[idx] == 1 else "SURVIVED"
    print(f"Patient (age {int(X_test.loc[idx, 'age'])}, actual outcome: {actual})")
    predict_patient(X_test.loc[idx], random_forest, scaled=False)
    print()


## Wrap Up and Next Steps

**What we built, start to finish:**
- Downloaded and verified a real clinical dataset of 299 heart failure patients
- Explored the data: class balance, feature distributions by outcome, correlations
- Built a proper train/test split with stratification, and scaled features correctly (fit on training data only)
- Trained and compared two models: an interpretable Logistic Regression baseline and a more powerful Random Forest
- Evaluated both with metrics appropriate for an imbalanced classification problem, not just accuracy
- Examined feature importance from both models to understand what drives the predictions
- Tested the model on individual patients

**Where to go from here:**
- Try other models (Gradient Boosting, XGBoost, a small neural network) and compare
- Address class imbalance directly with techniques like SMOTE or class weighting, and see how that changes recall on the minority (died) class
- Use cross-validation instead of a single train/test split for a more robust performance estimate, given the dataset's modest size
- Explore SHAP values for a more rigorous, per-prediction explanation than raw feature importances
